# ICS 591 Team 2 — Stage 3: Deploy the verification classifier (Render)

Converts the v1 model to **ONNX**, builds the `intelligence/` service to match `spec/classifier-contract.md`, and tests it end to end. Nothing is published until every check passes.

**Steps:**
1. Write the service code, then run **all tests** (contract, logic and HTTP) and the linter, exactly as CI does.
2. Export the model to ONNX. **Check:** ONNX outputs equal PyTorch, and the PyTorch-free preprocessing equals training preprocessing.
3. **Parity check:** the full service code reproduces the Stage 2c test results.
4. **End-to-end contract test** with the real model, including the security cases.
5. Upload the model to the **private** Hugging Face model repository. Generate the Render settings and tokens.
6. *(You, in the browser)* GitHub pull request, then the Render service.
7. Smoke-test the live service and save a deployment record (no secrets) to Drive.

**Before you start:** Colab **🔑 Secrets** `HF_WRITE` and `HF_READ` must be set, with notebook access on. Use a CPU runtime.

In [ ]:
# 1. Setup
!pip -q install timm onnx onnxruntime onnxscript fastapi python-multipart httpx huggingface_hub pytest ruff
import os, sys, json, time, hashlib, pathlib, secrets, io, shutil
import numpy as np
from google.colab import drive, userdata
drive.mount('/content/drive')
FT = pathlib.Path('/content/drive/MyDrive/ics591_finetune')
TEST_DIR = pathlib.Path('/content/drive/MyDrive/ics591_local_data/dataset_local/test')
for p in [FT / 'verifier_v1.pt', FT / 'verifier_v1_config.json', TEST_DIR]:
    assert p.exists(), f'missing: {p}'
SVC = pathlib.Path('/content/intelligence'); (SVC / 'tests').mkdir(parents=True, exist_ok=True)
DEPLOY = pathlib.Path('/content/deploy_model'); DEPLOY.mkdir(exist_ok=True)
os.chdir(SVC); sys.path.insert(0, str(SVC))
print('ok')

## 2. Service code (`intelligence/`)
Each cell writes one file. This folder is what goes to GitHub.

In [ ]:
%%writefile preprocess.py
"""Image preprocessing in plain numpy + PIL, reproducing the training pipeline
    torchvision: Resize(256) -> CenterCrop(224) -> ToTensor() -> Normalize(mean, std)
without needing torch at serving time. The deploy notebook checks that this gives
the same tensor as torchvision on real test images.
"""
import numpy as np
from PIL import Image


def resize_shorter_side(img, size):
    # torchvision rule: shorter side -> size, longer side -> int(size * long / short)
    w, h = img.size
    if w <= h:
        new_w, new_h = size, int(size * h / w)
    else:
        new_w, new_h = int(size * w / h), size
    return img.resize((new_w, new_h), Image.BILINEAR)


def center_crop(img, size):
    # torchvision rule: offsets are int(round((dim - size) / 2.0))
    w, h = img.size
    top = round((h - size) / 2.0)
    left = round((w - size) / 2.0)
    return img.crop((left, top, left + size, top + size))


def to_input(img, image_size, mean, std, resize_to=256):
    """RGB PIL image -> float32 array of shape (1, 3, image_size, image_size)."""
    img = center_crop(resize_shorter_side(img, resize_to), image_size)
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = (x - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None])

In [ ]:
%%writefile decision.py
"""Decision logic for the waste verification classifier (v1).

Pure numpy, no torch, so it can be unit-tested anywhere. It must stay identical
to the logic used when the model was evaluated (Stage 2c notebook, cell 6).
"""
import numpy as np

DECISIONS = ['accept', 'reject_contaminated', 'reject_wrong_type', 'retake_photo']
UNCERTAIN = 'uncertain_human_review'


def apply_temperature(p, t):
    """Rescale probabilities by temperature t (t < 1 sharpens, t > 1 softens)."""
    z = np.log(np.clip(p, 1e-12, 1)) / t
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def decide(pm, pc):
    """Combine the material head (plastic, non_plastic, discard) and the cleanliness
    head (clean, contaminated) into one decision index and a decision confidence."""
    mat = pm.argmax(1)
    cl = pc.argmax(1)
    dec = np.select([mat == 2, mat == 1, (mat == 0) & (cl == 1)], [3, 2, 1], default=0)
    conf = np.where(mat == 0, pm.max(1) * pc.max(1), pm.max(1))
    return dec, conf


def final_decision(pm, pc, threshold):
    """Single-image helper: returns (decision, suggested_decision, confidence, status)."""
    dec, conf = decide(pm, pc)
    suggested = DECISIONS[int(dec[0])]
    c = float(conf[0])
    if c >= threshold:
        return suggested, suggested, c, 'auto'
    return UNCERTAIN, suggested, c, 'review'

In [ ]:
%%writefile validation.py
"""Input validation for pickup photos and photo links. No web framework or torch needed.

Every image check raises ValidationError with a machine-readable code, so the API
can answer precisely and the monitor can count each failure type.
"""
import io
from urllib.parse import urlsplit

from PIL import Image, ImageOps, UnidentifiedImageError

MAX_BYTES = 8 * 1024 * 1024          # the Engine stores at most 5 MB; 8 MB leaves headroom
MIN_SIDE = 128                       # smaller than this carries too little detail to classify
MAX_PIXELS = 40_000_000              # guards against decompression bombs (a tiny file that expands hugely)
ALLOWED_FORMATS = {'JPEG', 'PNG', 'WEBP'}
TRAINING_MAX_SIDE = 768              # local training images were stored at <= 768 px ...
TRAINING_JPEG_QUALITY = 92           # ... as JPEG quality 92 (Stage 2a download step)
SIGNED_PATH_PREFIX = '/storage/v1/object/sign/'   # Supabase Storage signed links


class ValidationError(Exception):
    def __init__(self, code, detail):
        super().__init__(detail)
        self.code, self.detail = code, detail


def photo_url_allowed(url, allowed_hosts):
    """Only fetch https links to our own Supabase Storage signed-object path.
    Anything else (other hosts, internal addresses, credentials in the URL,
    odd ports, other paths) is refused, which blocks server-side request forgery."""
    try:
        p = urlsplit(url)
        port = p.port
    except ValueError:
        return False
    return (p.scheme == 'https' and p.hostname is not None and p.hostname.lower() in allowed_hosts
            and p.username is None and p.password is None and port in (None, 443)
            and p.path.startswith(SIGNED_PATH_PREFIX))


def load_image(data):
    """Validate raw image bytes and return an RGB PIL image prepared exactly the way
    training images were prepared (to avoid training/serving skew)."""
    if not data:
        raise ValidationError('empty_file', 'no image data received')
    if len(data) > MAX_BYTES:
        raise ValidationError('file_too_large', f'image larger than {MAX_BYTES // (1024 * 1024)} MB')
    try:
        img = Image.open(io.BytesIO(data))
        fmt = img.format
    except Image.DecompressionBombError:
        raise ValidationError('too_many_pixels', 'image dimensions are far beyond the limit') from None
    except (UnidentifiedImageError, OSError):
        raise ValidationError('unsupported_type', 'file is not a readable image') from None
    # The format comes from the file's actual bytes, never from a client header.
    if fmt not in ALLOWED_FORMATS:
        raise ValidationError('unsupported_type', f'format {fmt} not allowed; use JPEG, PNG or WEBP')
    w, h = img.size
    if w * h > MAX_PIXELS:
        raise ValidationError('too_many_pixels', f'image has {w * h} pixels; limit is {MAX_PIXELS}')
    if min(w, h) < MIN_SIDE:
        raise ValidationError('image_too_small', f'shortest side is {min(w, h)} px; minimum is {MIN_SIDE}')
    try:
        img.load()
    except Exception:
        raise ValidationError('corrupt_image', 'image data is truncated or corrupt') from None

    img = ImageOps.exif_transpose(img)      # phones store rotation in EXIF; apply it
    if img.mode in ('P', 'LA', 'RGBA'):
        img = img.convert('RGBA').convert('RGB')
    else:
        img = img.convert('RGB')
    return normalise_like_training(img)


def normalise_like_training(img):
    """Training images were downscaled to <= 768 px and re-saved as JPEG q92.
    Doing the same here keeps serving inputs statistically like training inputs.
    Re-encoding also drops all metadata (including any GPS location in EXIF)."""
    s = TRAINING_MAX_SIDE / max(img.size)
    if s < 1:
        img = img.resize((round(img.width * s), round(img.height * s)), Image.BICUBIC)
    buf = io.BytesIO()
    img.save(buf, 'JPEG', quality=TRAINING_JPEG_QUALITY)
    buf.seek(0)
    return Image.open(buf).convert('RGB')


def dhash(img, size=8):
    """64-bit difference hash: near-identical photos (resized, recompressed) get
    hashes that differ in only a few bits. Used to monitor reused pickup photos."""
    g = img.convert('L').resize((size + 1, size), Image.BILINEAR)
    px = list(g.getdata())
    bits = 0
    for row in range(size):
        for col in range(size):
            left = px[row * (size + 1) + col]
            right = px[row * (size + 1) + col + 1]
            bits = (bits << 1) | (left > right)
    return bits

In [ ]:
%%writefile security.py
"""Bearer-token authentication, roles and per-token rate limiting.

Tokens are never stored in code or files. They come from the SERVICE_TOKENS
environment variable (a Hugging Face Space secret), in the form
    name:role:token;name:role:token
Roles:
    backend  -> may call /v1/verify (the Engine)
    reviewer -> /v1/verify, /v1/verify-upload, /metrics and /v1/logs (paired-team test accounts)
    admin    -> everything
"""
import hashlib
import hmac
import threading
import time
from collections import defaultdict, deque

PERMISSIONS = {
    'backend': {'verify'},
    'reviewer': {'verify', 'upload', 'metrics', 'logs'},
    'admin': {'verify', 'upload', 'metrics', 'logs'},
}


class AuthError(Exception):
    def __init__(self, status, code):
        super().__init__(code)
        self.status, self.code = status, code


def parse_tokens(spec):
    tokens = []
    for part in (spec or '').split(';'):
        part = part.strip()
        if not part:
            continue
        name, role, token = part.split(':', 2)
        if role not in PERMISSIONS:
            raise ValueError(f'unknown role {role!r} for {name!r}')
        if len(token) < 24:
            raise ValueError(f'token for {name!r} is too short (need >= 24 characters)')
        tokens.append({'name': name, 'role': role, 'digest': hashlib.sha256(token.encode()).digest()})
    return tokens


def authenticate(tokens, authorization_header, permission):
    """Returns the caller's name, or raises AuthError(401/403)."""
    if not authorization_header or not authorization_header.startswith('Bearer '):
        raise AuthError(401, 'unauthorised')
    presented = hashlib.sha256(authorization_header[7:].strip().encode()).digest()
    caller = None
    for t in tokens:                       # compare against every token in constant time
        if hmac.compare_digest(presented, t['digest']):
            caller = t
    if caller is None:
        raise AuthError(401, 'unauthorised')
    if permission not in PERMISSIONS[caller['role']]:
        raise AuthError(403, 'forbidden')
    return caller['name']


class RateLimiter:
    def __init__(self, per_minute=60):
        self.per_minute = per_minute
        self.hits = defaultdict(deque)
        self.lock = threading.Lock()

    def check(self, key, now=None):
        """Returns 0 if allowed, otherwise the number of seconds to wait."""
        now = time.time() if now is None else now
        with self.lock:
            q = self.hits[key]
            while q and now - q[0] >= 60:
                q.popleft()
            if len(q) >= self.per_minute:
                return int(60 - (now - q[0])) + 1
            q.append(now)
            return 0

In [ ]:
%%writefile monitoring.py
"""Live monitoring hooks for the verification classifier (signal S1 of the M1 spec,
plus operational signals). Every request becomes an event; /metrics summarises a
time window and raises alerts. Events are also appended to a JSON-lines log.

Never logged: images, photo links (they carry a signing token), or personal data.
"""
import json
import math
import os
import threading
import time
from collections import Counter, deque

ALERTS = {
    # Engine policy (backend/README.md): accept needs >= 0.90, reject >= 0.80; the rest
    # goes to the rider. A high deferred share means the model is rarely sure.
    'deferred_share_max': 0.70,
    # Share of photos we answer 422 (unusable -> retake). Offline, 33% of web images were junk;
    # real pickup photos should be cleaner.
    'unusable_share_max': 0.30,
    # Near-duplicate photos across different verifications: a reuse/fraud signal.
    'duplicate_rate_max': 0.05,
    # The Engine gives up after 8 s; stay well inside that.
    'latency_p95_ms_max': 6000,
    # Population stability index of the confidence distribution vs the offline baseline.
    'confidence_psi_warn': 0.10,
    'confidence_psi_alert': 0.20,
}
BINS = 10
ENGINE_ACCEPT_AT = float(os.environ.get('ENGINE_ACCEPT_AT', '0.90'))
ENGINE_REJECT_AT = float(os.environ.get('ENGINE_REJECT_AT', '0.80'))


def psi(live_counts, baseline_shares, eps=1e-4):
    """PSI = sum over bins of (a - e) * ln(a / e); a = live share, e = baseline share."""
    n = sum(live_counts)
    if n == 0 or not baseline_shares or len(baseline_shares) != len(live_counts):
        return None
    total = 0.0
    for c, e in zip(live_counts, baseline_shares, strict=True):
        a = max(c / n, eps)
        e = max(e, eps)
        total += (a - e) * math.log(a / e)
    return total


def percentile(xs, q):
    if not xs:
        return None
    xs = sorted(xs)
    k = (len(xs) - 1) * q
    lo, hi = math.floor(k), math.ceil(k)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


def histogram(confs):
    hist = [0] * BINS
    for c in confs:
        hist[min(int(c * BINS), BINS - 1)] += 1
    return hist


def deferred_by_engine(event):
    """Would the Engine's policy send this decision to the rider?"""
    if event['decision'] == 'accept':
        return event['confidence'] < ENGINE_ACCEPT_AT
    if event['decision'] == 'reject':
        return event['confidence'] < ENGINE_REJECT_AT
    return True


class Monitor:
    def __init__(self, log_path=None, baseline_hist=None, max_events=20000, dup_hamming=5):
        self.events = deque(maxlen=max_events)
        self.log_path = log_path
        self.baseline = baseline_hist
        self.started = time.time()
        self.lock = threading.Lock()
        self.seen = deque(maxlen=50000)        # (sha256, dhash, verification_id)
        self.dup_hamming = dup_hamming

    def check_duplicate(self, sha, dh, verification_id):
        """Earlier verification (a different id) with the same or a near-identical photo.
        A retry of the same verification is not a duplicate."""
        with self.lock:
            match = None
            for prev_sha, prev_dh, prev_id in self.seen:
                if prev_id == verification_id:
                    continue
                if prev_sha == sha or (prev_dh ^ dh).bit_count() <= self.dup_hamming:
                    match = prev_id
                    break
            self.seen.append((sha, dh, verification_id))
        return match

    def record(self, event):
        event = dict(event, ts=time.time())
        with self.lock:
            self.events.append(event)
            if self.log_path:
                with open(self.log_path, 'a') as f:
                    f.write(json.dumps(event, default=str) + '\n')

    def recent(self, limit=100):
        with self.lock:
            return list(self.events)[-limit:]

    def metrics(self, window_hours=24.0):
        cutoff = time.time() - window_hours * 3600
        with self.lock:
            ev = [e for e in self.events if e['ts'] >= cutoff]
        ok = [e for e in ev if e.get('outcome') == 'predicted']
        unusable = [e for e in ev if e.get('outcome') == 'unusable_photo']
        answered = len(ok) + len(unusable)
        confs = [e['confidence'] for e in ok]
        hist = histogram(confs)
        per_type = {}
        for t in sorted({e.get('declared_type') or 'unknown' for e in ok}):
            sub = [e['confidence'] for e in ok if (e.get('declared_type') or 'unknown') == t]
            per_type[t] = {'n': len(sub), 'mean': round(sum(sub) / len(sub), 4), 'histogram': histogram(sub)}
        lat = [e['latency_ms'] for e in ok]
        model_lat = [e['model_ms'] for e in ok if 'model_ms' in e]
        p = psi(hist, self.baseline) if ok and self.baseline else None
        m = {
            'window_hours': window_hours,
            'uptime_hours': round((time.time() - self.started) / 3600, 2),
            'requests_total': len(ev),
            'outcomes': dict(Counter(e.get('outcome') for e in ev)),
            'predictions': len(ok),
            'decisions': dict(Counter(f"{e['decision']}/{e['reason']}" for e in ok)),
            'deferred_share_under_engine_policy':
                round(sum(deferred_by_engine(e) for e in ok) / len(ok), 4) if ok else None,
            'unusable_share': round(len(unusable) / answered, 4) if answered else None,
            'unusable_reasons': dict(Counter(e.get('error') for e in unusable)),
            'confidence_mean': round(sum(confs) / len(confs), 4) if confs else None,
            'confidence_histogram': {'bins': [round(i / BINS, 1) for i in range(BINS + 1)], 'counts': hist},
            'confidence_by_declared_type': per_type,
            'confidence_psi_vs_offline': round(p, 4) if p is not None else None,
            'duplicate_rate': round(sum(bool(e.get('duplicate_of')) for e in ok) / len(ok), 4) if ok else None,
            'latency_ms_p50': round(percentile(lat, 0.5), 1) if lat else None,
            'latency_ms_p95': round(percentile(lat, 0.95), 1) if lat else None,
            'model_ms_p95': round(percentile(model_lat, 0.95), 1) if model_lat else None,
        }
        m['alerts'] = self.alerts(m)
        m['thresholds'] = dict(ALERTS, engine_accept_at=ENGINE_ACCEPT_AT, engine_reject_at=ENGINE_REJECT_AT)
        return m

    @staticmethod
    def alerts(m):
        out = []

        def over(key, limit, label):
            if m.get(key) is not None and m[key] > limit:
                out.append(f'{label}: {m[key]} > {limit}')

        over('deferred_share_under_engine_policy', ALERTS['deferred_share_max'], 'most decisions deferred to riders')
        over('unusable_share', ALERTS['unusable_share_max'], 'many unusable photos')
        over('duplicate_rate', ALERTS['duplicate_rate_max'], 'duplicate photos high')
        over('latency_ms_p95', ALERTS['latency_p95_ms_max'], 'latency p95 near the Engine timeout')
        p = m.get('confidence_psi_vs_offline')
        if p is not None and p > ALERTS['confidence_psi_alert']:
            out.append(f'confidence drift ALERT: PSI {p} > {ALERTS["confidence_psi_alert"]}')
        elif p is not None and p > ALERTS['confidence_psi_warn']:
            out.append(f'confidence drift warning: PSI {p} > {ALERTS["confidence_psi_warn"]}')
        blocked = m['outcomes'].get('url_blocked', 0)
        if blocked:
            out.append(f'security: {blocked} request(s) asked for a photo link outside our storage')
        if m['outcomes'].get('auth_failed', 0) >= 10:
            out.append(f"security: {m['outcomes']['auth_failed']} failed authentications")
        if m['predictions'] < 30:
            out.append(f"note: only {m['predictions']} predictions in window; rates are not yet reliable")
        return out

In [ ]:
%%writefile service.py
"""The classifier's request handling, independent of the web framework, so every
rule in spec/classifier-contract.md can be tested without a server or a model.

Status codes follow the contract exactly:
    200  decision
    422  ONLY when the photo itself can't be used -> the Engine asks for a retake
    400  malformed request or a photo link outside our storage -> Engine treats as unavailable
    401  wrong or missing token; 403 token lacks the permission
    429  rate limited (the Engine retries once)
    503  the photo couldn't be fetched
    500  unexpected failure
"""
import hashlib
import json
import threading
import time
import traceback
import uuid
from datetime import datetime
from typing import Literal
from uuid import UUID

from pydantic import BaseModel, ConfigDict, Field
from pydantic import ValidationError as SchemaError

from security import AuthError, authenticate
from validation import ValidationError, dhash, load_image, photo_url_allowed

CONTRACT = {
    'accept': ('accept', 'ok'),
    'reject_contaminated': ('reject', 'contaminated'),
    'reject_wrong_type': ('reject', 'wrong_type'),
}


class PhotoFetchError(Exception):
    """The photo link could not be fetched (timeout, connection error, non-200)."""


class PhotoTooLarge(Exception):
    """The fetched photo exceeded the size limit."""


class PhotoRef(BaseModel):
    model_config = ConfigDict(extra='forbid')
    url: str = Field(min_length=1, max_length=4096)
    sha256: str = Field(pattern=r'^[0-9a-f]{64}$')


class VerifyBody(BaseModel):
    # extra='forbid' enforces the boundary: a request carrying any other field
    # (a name, a phone number, a location...) is refused, not silently accepted.
    model_config = ConfigDict(extra='forbid')
    verification_id: UUID
    request_id: UUID
    declared_type: Literal['pet', 'hdpe', 'pp', 'mixed_plastic']
    photo: PhotoRef
    captured_at: datetime
    capture_source: Literal['rider', 'household']


class IdempotencyCache:
    """Remembers recent answers by verification_id so an Engine retry gets the same
    answer without running the model again (the free server's CPU is slow)."""

    def __init__(self, ttl_s=600, max_items=2000):
        self.ttl_s, self.max_items = ttl_s, max_items
        self.items = {}
        self.lock = threading.Lock()

    def get(self, key):
        with self.lock:
            hit = self.items.get(key)
            if hit and time.time() - hit[0] < self.ttl_s:
                return hit[1], hit[2]
            return None

    def put(self, key, status, body):
        with self.lock:
            if len(self.items) >= self.max_items:
                del self.items[min(self.items, key=lambda k: self.items[k][0])]
            self.items[key] = (time.time(), status, body)


class ClassifierService:
    def __init__(self, verifier, monitor, tokens, limiter, fetch_photo, allowed_hosts):
        self.verifier = verifier
        self.monitor = monitor
        self.tokens = tokens
        self.limiter = limiter
        self.fetch_photo = fetch_photo
        self.allowed_hosts = {h.strip().lower() for h in allowed_hosts if h.strip()}
        self.cache = IdempotencyCache()

    # ---------------------------------------------------------------- helpers
    def _guard(self, authorization, permission):
        try:
            caller = authenticate(self.tokens, authorization, permission)
        except AuthError as e:
            self.monitor.record({'outcome': 'auth_failed', 'error': e.code, 'endpoint': permission})
            return None, (e.status, {'error': e.code})
        wait = self.limiter.check(caller)
        if wait:
            self.monitor.record({'outcome': 'rate_limited', 'caller': caller, 'endpoint': permission})
            return None, (429, {'error': 'rate_limited', 'retry_after_s': wait})
        return caller, None

    def _classify(self, data, verification_id, caller, extra_log, t0):
        """Shared by the contract endpoint and the test-upload endpoint.
        Returns (status, body, detail) where detail holds the head probabilities."""
        try:
            img = load_image(data)
        except ValidationError as e:
            self.monitor.record({'outcome': 'unusable_photo', 'error': e.code, 'caller': caller, **extra_log})
            return 422, {'error': 'photo_unusable', 'reason': e.code, 'detail': e.detail}, None
        sha = hashlib.sha256(data).hexdigest()
        dup = self.monitor.check_duplicate(sha, dhash(img), verification_id)
        m0 = time.perf_counter()
        out = self.verifier.predict(img)
        model_ms = round((time.perf_counter() - m0) * 1000, 1)
        latency = round((time.perf_counter() - t0) * 1000, 1)
        suggested = out['suggested_decision']
        if suggested == 'retake_photo':
            self.monitor.record({'outcome': 'unusable_photo', 'error': 'nothing_recognisable', 'caller': caller,
                                 'confidence': out['confidence'], 'latency_ms': latency, **extra_log})
            return 422, {'error': 'photo_unusable', 'reason': 'nothing_recognisable',
                         'detail': 'no recognisable waste in frame; please retake'}, out
        decision, reason = CONTRACT[suggested]
        body = {'verification_id': str(verification_id), 'decision': decision, 'confidence': out['confidence'],
                'reason': reason, 'model_version': self.verifier.version}
        self.monitor.record({'outcome': 'predicted', 'caller': caller, 'decision': decision, 'reason': reason,
                             'confidence': out['confidence'], 'image_sha256': sha, 'duplicate_of': dup,
                             'latency_ms': latency, 'model_ms': model_ms, 'model_version': self.verifier.version,
                             **extra_log})
        return 200, body, dict(out, duplicate_of=dup)

    # -------------------------------------------------------------- endpoints
    def health(self):
        return 200, {'status': 'ok', 'model_version': self.verifier.version,
                     'weights_sha256': self.verifier.weights_sha256}

    def verify_json(self, authorization, idempotency_key, raw_body):
        """POST /v1/verify, exactly as in spec/classifier-contract.md."""
        t0 = time.perf_counter()
        caller, err = self._guard(authorization, 'verify')
        if err:
            return err
        try:
            body = VerifyBody.model_validate(json.loads(raw_body or b'null'))
        except json.JSONDecodeError:
            self.monitor.record({'outcome': 'bad_request', 'error': 'invalid_json', 'caller': caller})
            return 400, {'error': 'invalid_json'}
        except SchemaError as e:
            fields = sorted({'.'.join(str(p) for p in err['loc']) for err in e.errors()})
            self.monitor.record({'outcome': 'bad_request', 'error': 'invalid_request', 'fields': fields,
                                 'caller': caller})
            return 400, {'error': 'invalid_request', 'fields': fields}
        vid = str(body.verification_id)
        if idempotency_key and idempotency_key != vid:
            self.monitor.record({'outcome': 'bad_request', 'error': 'idempotency_key_mismatch', 'caller': caller})
            return 400, {'error': 'idempotency_key_mismatch'}
        cached = self.cache.get(vid)
        if cached:
            self.monitor.record({'outcome': 'idempotent_replay', 'caller': caller, 'verification_id': vid})
            return cached
        log = {'verification_id': vid, 'request_id': str(body.request_id), 'declared_type': body.declared_type,
               'capture_source': body.capture_source, 'captured_at': body.captured_at.isoformat()}
        if not photo_url_allowed(body.photo.url, self.allowed_hosts):
            self.monitor.record({'outcome': 'url_blocked', 'caller': caller, **log})
            return 400, {'error': 'photo_url_not_allowed'}
        try:
            data = self.fetch_photo(body.photo.url)
        except PhotoTooLarge:
            self.monitor.record({'outcome': 'unusable_photo', 'error': 'file_too_large', 'caller': caller, **log})
            return 422, {'error': 'photo_unusable', 'reason': 'file_too_large'}
        except PhotoFetchError as e:
            self.monitor.record({'outcome': 'photo_unavailable', 'error': str(e)[:80], 'caller': caller, **log})
            return 503, {'error': 'photo_unavailable'}
        if hashlib.sha256(data).hexdigest() != body.photo.sha256:
            self.monitor.record({'outcome': 'unusable_photo', 'error': 'photo_hash_mismatch', 'caller': caller, **log})
            return 422, {'error': 'photo_unusable', 'reason': 'photo_hash_mismatch'}
        try:
            status, resp, _ = self._classify(data, vid, caller, log, t0)
        except Exception:
            return self._internal(caller, log)
        if status in (200, 422):
            self.cache.put(vid, status, resp)
        return status, resp

    def verify_upload(self, authorization, data, declared_type=None):
        """POST /v1/verify-upload: test and demo path for reviewers and admins only.
        Same model and checks, but the photo is uploaded directly instead of linked."""
        t0 = time.perf_counter()
        caller, err = self._guard(authorization, 'upload')
        if err:
            return err
        vid = str(uuid.uuid4())
        log = {'verification_id': vid, 'declared_type': declared_type, 'capture_source': 'test_upload'}
        try:
            status, resp, detail = self._classify(data, vid, caller, log, t0)
        except Exception:
            return self._internal(caller, log)
        if detail:
            resp = dict(resp, material=detail['material'], cleanliness=detail['cleanliness'],
                        duplicate_of=detail['duplicate_of'])
        return status, resp

    def metrics(self, authorization, window_hours=24.0):
        _, err = self._guard(authorization, 'metrics')
        if err:
            return err
        return 200, dict(self.monitor.metrics(window_hours), model_version=self.verifier.version)

    def logs(self, authorization, limit=100):
        _, err = self._guard(authorization, 'logs')
        if err:
            return err
        return 200, {'events': self.monitor.recent(limit)}

    def _internal(self, caller, log):
        ref = uuid.uuid4().hex[:12]
        print(f'[internal_error {ref}]', traceback.format_exc(), flush=True)
        self.monitor.record({'outcome': 'internal_error', 'caller': caller, 'ref': ref, **log})
        return 500, {'error': 'internal', 'ref': ref}

In [ ]:
%%writefile photo_fetch.py
"""Fetches a photo from a signed Supabase Storage link.

Defences: the caller has already checked the link against the allow-list
(validation.photo_url_allowed); redirects are NOT followed (a redirect could point
anywhere); the download is streamed and cut off at the size limit; and the whole
fetch has a short timeout so we stay inside the Engine's 8-second budget.
"""
import httpx

from service import PhotoFetchError, PhotoTooLarge
from validation import MAX_BYTES


def make_fetcher(timeout_s=4.0, max_bytes=MAX_BYTES, transport=None):
    client = httpx.Client(timeout=timeout_s, follow_redirects=False, transport=transport)

    def fetch(url):
        try:
            with client.stream('GET', url) as r:
                if r.status_code != 200:
                    raise PhotoFetchError(f'HTTP {r.status_code}')
                buf = bytearray()
                for chunk in r.iter_bytes():
                    buf += chunk
                    if len(buf) > max_bytes:
                        raise PhotoTooLarge()
                return bytes(buf)
        except httpx.HTTPError as e:
            raise PhotoFetchError(type(e).__name__) from None

    return fetch

In [ ]:
%%writefile verifier.py
"""Loads the v1 verification model (ONNX) and runs one prediction.

The model was trained in PyTorch (Stage 2c) and exported to ONNX so that the
service runs without PyTorch and fits in a small (512 MB) server. The deploy
notebook checks that the ONNX model and this preprocessing reproduce the
offline results before anything is published.
"""
import hashlib
import json
import os

import numpy as np
import onnxruntime as ort

from decision import apply_temperature, final_decision
from preprocess import to_input

WEIGHTS_FILE = 'verifier_v1.onnx'
CONFIG_FILE = 'verifier_v1_config.json'


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def fetch_model_files():
    """Uses local files if MODEL_DIR is set; otherwise downloads from the private
    Hugging Face model repo named in MODEL_REPO, pinned to MODEL_REVISION."""
    local = os.environ.get('MODEL_DIR')
    if local:
        return os.path.join(local, WEIGHTS_FILE), os.path.join(local, CONFIG_FILE)
    from huggingface_hub import hf_hub_download
    repo = os.environ['MODEL_REPO']
    rev = os.environ.get('MODEL_REVISION') or None
    tok = os.environ.get('HF_TOKEN')
    return (hf_hub_download(repo, WEIGHTS_FILE, revision=rev, token=tok),
            hf_hub_download(repo, CONFIG_FILE, revision=rev, token=tok))


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


class Verifier:
    def __init__(self, weights_path, config_path, expected_sha256=None):
        with open(config_path) as f:
            self.cfg = json.load(f)
        self.weights_sha256 = sha256_file(weights_path)
        if expected_sha256 and expected_sha256 != self.weights_sha256:
            raise RuntimeError('model weights hash does not match EXPECTED_WEIGHTS_SHA256 - refusing to serve')
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = int(os.environ.get('ORT_THREADS', '1'))
        self.session = ort.InferenceSession(weights_path, sess_options=opts, providers=['CPUExecutionProvider'])
        self.input_name = self.session.get_inputs()[0].name
        self.version = f"verifier-v1+{self.weights_sha256[:12]}"
        self.threshold = float(self.cfg['auto_decision_threshold'])

    def logits(self, img):
        x = to_input(img, self.cfg['image_size'], self.cfg['normalize_mean'], self.cfg['normalize_std'])
        om, oc = self.session.run(None, {self.input_name: x})
        return om, oc

    def predict(self, img):
        om, oc = self.logits(img)
        pm = apply_temperature(softmax(om), self.cfg['temperature_material'])
        pc = apply_temperature(softmax(oc), self.cfg['temperature_clean'])
        decision, suggested, conf, status = final_decision(pm, pc, self.threshold)
        return {
            'decision': decision,
            'suggested_decision': suggested,
            'confidence': round(conf, 4),
            'status': status,
            'material': dict(zip(self.cfg['material_classes'], np.round(pm[0], 4).tolist(), strict=True)),
            'cleanliness': dict(zip(self.cfg['clean_classes'], np.round(pc[0], 4).tolist(), strict=True)),
        }

In [ ]:
%%writefile main.py
"""Waste verification classifier: the web layer. All rules live in service.py;
this file only maps HTTP requests to it. Contract: spec/classifier-contract.md.

    GET  /v1/health        200 once the model is loaded (Render health check)
    POST /v1/verify        the Engine's call (JSON with a signed photo link)
    POST /v1/verify-upload test/demo: upload a photo directly (reviewer, admin)
    GET  /metrics          live monitoring hooks and alerts (reviewer, admin)
    GET  /v1/logs          recent request events, no images or links (reviewer, admin)
"""
import os
import pathlib
from contextlib import asynccontextmanager
from typing import Annotated

from fastapi import FastAPI, File, Form, Header, Query, Request, UploadFile
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from starlette.concurrency import run_in_threadpool

from monitoring import Monitor
from security import RateLimiter, parse_tokens
from service import ClassifierService
from validation import MAX_BYTES

STATE = {}


@asynccontextmanager
async def lifespan(app):
    from photo_fetch import make_fetcher
    from verifier import Verifier, fetch_model_files

    tokens = parse_tokens(os.environ.get('SERVICE_TOKENS', ''))
    if not tokens:
        raise RuntimeError('SERVICE_TOKENS is not set - refusing to start without authentication')
    hosts = os.environ.get('PHOTO_URL_ALLOWED_HOSTS', '').split(',')
    if not any(h.strip() for h in hosts):
        raise RuntimeError('PHOTO_URL_ALLOWED_HOSTS is not set - refusing to fetch photos from anywhere')
    w, c = fetch_model_files()
    verifier = Verifier(w, c, os.environ.get('EXPECTED_WEIGHTS_SHA256') or None)
    log_dir = pathlib.Path(os.environ.get('LOG_DIR', 'logs'))
    log_dir.mkdir(parents=True, exist_ok=True)
    monitor = Monitor(log_path=log_dir / 'requests.jsonl', baseline_hist=verifier.cfg.get('baseline_confidence_shares'))
    STATE['svc'] = ClassifierService(verifier, monitor, tokens,
                                     RateLimiter(int(os.environ.get('RATE_LIMIT_PER_MIN', '60'))),
                                     make_fetcher(), hosts)
    yield


app = FastAPI(title='Waste Verification Classifier', version='1.0', lifespan=lifespan,
              docs_url='/docs' if os.environ.get('ENABLE_DOCS', '1') == '1' else None, redoc_url=None)


def reply(result):
    status, body = result
    return JSONResponse(status_code=status, content=body, headers={'Cache-Control': 'no-store'})


@app.exception_handler(RequestValidationError)
async def bad_request(request, exc):
    # FastAPI's default is 422, which the Engine reads as "retake the photo". A malformed
    # request is not a photo problem, so answer 400 instead.
    return reply((400, {'error': 'invalid_request'}))


@app.get('/v1/health')
@app.get('/health')
def health():
    return reply(STATE['svc'].health())


@app.post('/v1/verify')
async def verify(request: Request, authorization: Annotated[str | None, Header()] = None,
                 idempotency_key: Annotated[str | None, Header()] = None):
    raw = await request.body()
    return reply(await run_in_threadpool(STATE['svc'].verify_json, authorization, idempotency_key, raw))


@app.post('/v1/verify-upload')
async def verify_upload(image: Annotated[UploadFile, File()],
                        declared_type: Annotated[str | None, Form()] = None,
                        authorization: Annotated[str | None, Header()] = None):
    data = await image.read(MAX_BYTES + 1)
    return reply(await run_in_threadpool(STATE['svc'].verify_upload, authorization, data, declared_type))


@app.get('/metrics')
def metrics(window_hours: Annotated[float, Query(gt=0, le=24 * 30)] = 24.0,
            authorization: Annotated[str | None, Header()] = None):
    return reply(STATE['svc'].metrics(authorization, window_hours))


@app.get('/v1/logs')
def logs(limit: Annotated[int, Query(ge=1, le=1000)] = 100,
         authorization: Annotated[str | None, Header()] = None):
    return reply(STATE['svc'].logs(authorization, limit))

In [ ]:
%%writefile requirements.txt
onnxruntime==1.19.2
fastapi==0.115.0
uvicorn==0.30.6
python-multipart==0.0.9
httpx==0.27.2
pydantic==2.9.2
pillow==10.4.0
numpy==1.26.4
huggingface_hub==0.25.2

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

# Run as an unprivileged user, never as root
RUN useradd -m -u 1000 app
WORKDIR /home/app/service

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Service code only: tests, notebooks and docs stay out of the image (see .dockerignore)
COPY *.py ./
USER app
ENV HF_HOME=/home/app/.cache/huggingface \
    LOG_DIR=/tmp/verifier-logs \
    PYTHONUNBUFFERED=1

# Render sets PORT (default 10000)
CMD ["sh", "-c", "uvicorn main:app --host 0.0.0.0 --port ${PORT:-10000}"]

In [ ]:
%%writefile .dockerignore
tests/
notebooks/
*.md
*.onnx
*.pt
.env

In [ ]:
%%writefile .gitignore
# secrets: never commit
.env
*.env
# model files live in the private Hugging Face model repo, not in git
*.onnx
*.pt
# runtime output
logs/
__pycache__/
.pytest_cache/
.ruff_cache/

In [ ]:
%%writefile ruff.toml
# Same rule families as backend/pyproject.toml, pinned so a new ruff release
# can't change what CI enforces. Notebooks are research records, not service code.
line-length = 120
target-version = "py311"
extend-exclude = ["notebooks"]

[lint]
select = ["E", "F", "I", "B", "UP", "S"]

[lint.per-file-ignores]
"tests/**" = ["S101", "S105", "S106", "S311", "E402"]

In [ ]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests

In [ ]:
%%writefile README.md
# Intelligence

The waste verification classifier, the system's one learned component. The Engine calls it over HTTP
as set out in [`../spec/classifier-contract.md`](../spec/classifier-contract.md). It receives a photo link and
request metadata, and returns a decision and a confidence. It never sees identity, location or payment data, and
it never moves money: the Engine's policy decides.

## The model (v1)

| | |
|---|---|
| Architecture | ViT-S/16 backbone, chosen over MobileNetV3, EfficientNet-B0, ResNet-50 and ConvNeXt-Tiny (Stage 1 notebooks), with two heads: **material** (plastic / non_plastic / discard) and **cleanliness** (clean / contaminated) |
| Training data | RealWaste (4,752 images, CC BY 4.0) plus human-labelled web images from Ghana (Stage 2 notebooks). Research use only; images are not in this repo |
| Served as | ONNX (exported from PyTorch; the deploy notebook checks the outputs match), so the service needs no PyTorch and fits a 512 MB free instance |
| Registry | Private Hugging Face model repo. The service downloads it at start-up, **pinned to a commit** and **checked by SHA-256** |
| Local test results | Pay/no-pay decision 91.2% correct; clean plastic recall 100% (14/14); exact 4-way decision 60.2% (n = 113) |
| Intended use | Assistant to a human verifier during the pilot. Not for unsupervised payouts |

How the model output becomes a contract reply:

| Model says | Reply |
|---|---|
| plastic + clean | `200` `accept` / `ok` |
| plastic + contaminated | `200` `reject` / `contaminated` |
| non-plastic | `200` `reject` / `wrong_type` |
| discard (no recognisable waste) | `422`: the Engine asks for a retake |

`confidence` is temperature-calibrated. The Engine applies its own thresholds (accept at 0.90 or above, reject at 0.80 or above).
`duplicate` is never sent. The Engine already rejects identical and resized photos, and a perceptual hash would risk
rejecting honest households whose sacks look alike. Reuse is still **measured** here (`duplicate_rate` in `/metrics`).

## Endpoints

| Method | Path | Caller | |
|---|---|---|---|
| GET | `/v1/health` | anyone | 200 once the model is loaded (Render health check) |
| POST | `/v1/verify` | Engine (`backend` token) | The contract call |
| POST | `/v1/verify-upload` | `reviewer`, `admin` | Test and demo: upload a photo directly (multipart field `image`, optional `declared_type`) |
| GET | `/metrics?window_hours=24` | `reviewer`, `admin` | Monitoring hooks and alerts |
| GET | `/v1/logs?limit=100` | `reviewer`, `admin` | Recent events (never images, photo links or personal data) |

Status codes: `200` decision · `422` **only** for an unusable photo (unreadable, too small, hash mismatch, nothing
recognisable) · `400` malformed request, unknown field, or a photo link outside our storage · `401`/`403` auth ·
`429` rate limit · `503` photo couldn't be fetched · `500` internal (the response includes a `ref` that matches the server log).

## Security controls

- **Tokens per caller** (`backend`, `reviewer1-3`, `admin`), compared in constant time; roles limit what each token can do.
- **Photo links are allow-listed:** https only, our Supabase host only, `/storage/v1/object/sign/` paths only, no
  credentials or odd ports, redirects not followed, downloads capped at 8 MB with a 4 s timeout. This blocks server-side request forgery.
- **The boundary is enforced:** requests with any field outside the contract (for example a phone number) are refused with `400`.
- **Images are checked** by their actual bytes (format, size, pixel count against decompression bombs, EXIF rotation),
  then re-encoded, which drops all metadata, including GPS.
- **Model integrity:** the service refuses to start if the downloaded weights don't match `EXPECTED_WEIGHTS_SHA256`.
- **Idempotency:** an Engine retry (same `Idempotency-Key`) gets the cached answer; the key must equal `verification_id`.
- **Rate limit:** 60 requests per minute per token. Non-root container. No secrets in this repo.

## Monitoring (M1 signal S1 and operational hooks)

`/metrics` reports, per time window:
- the confidence histogram, overall and **per declared waste type** (S1), with a **PSI drift score** against the offline test baseline,
- the decision mix and the share the Engine policy would defer to riders,
- the unusable-photo share and its reasons,
- duplicate rate, and p50/p95 latency (total and model-only),
- security events (blocked links, failed authentications),
- alerts whenever a threshold is crossed (thresholds are in `monitoring.py`, each with its reason).

## Configuration (Render environment variables; never commit these)

| Variable | Purpose |
|---|---|
| `SERVICE_TOKENS` | `name:role:token;...` (roles: `backend`, `reviewer`, `admin`); the service won't start without it |
| `PHOTO_URL_ALLOWED_HOSTS` | Supabase project host, for example `abcd.supabase.co`; the service won't start without it |
| `MODEL_REPO`, `MODEL_REVISION` | Private Hugging Face model repo and the commit to serve |
| `HF_TOKEN` | **Read-only** Hugging Face token for the private model |
| `EXPECTED_WEIGHTS_SHA256` | SHA-256 of `verifier_v1.onnx` |
| `ORT_THREADS`, `RATE_LIMIT_PER_MIN`, `ENABLE_DOCS` | Optional (1, 60, 1) |

## Layout

| Path | What |
|---|---|
| `main.py` | Web layer only (FastAPI) |
| `service.py` | Every contract rule: auth, schema, idempotency, link allow-list, fetch, hash check, decision mapping |
| `verifier.py`, `preprocess.py`, `decision.py` | ONNX model, preprocessing identical to training, decision logic |
| `validation.py`, `security.py`, `monitoring.py`, `photo_fetch.py` | Input checks, tokens and rate limit, metrics, safe photo download |
| `tests/` | Contract, logic and HTTP tests (no model or network needed) |
| `notebooks/` | The research record: model selection, data collection, labelling, fine-tuning, deployment |

## What CI expects

- `requirements.txt` in this folder (installed with `uv pip`).
- Tests under `tests/`, run with `pytest`. Code must pass `ruff check` (rules pinned in `ruff.toml`).
- No model weights or datasets in Git. They live in the private model repo and are downloaded at start-up.

Run locally: `pip install -r requirements.txt pytest && pytest && ruff check .`

In [ ]:
%%writefile tests/__init__.py

In [ ]:
%%writefile tests/test_logic.py
"""Unit tests for the parts that need neither a model nor a web server."""
import io

import numpy as np
import pytest
from PIL import Image

from decision import DECISIONS, UNCERTAIN, apply_temperature, decide, final_decision
from monitoring import Monitor, psi
from preprocess import center_crop, resize_shorter_side, to_input
from security import AuthError, RateLimiter, authenticate, parse_tokens
from validation import ValidationError, dhash, load_image, photo_url_allowed


def image_bytes(w=640, h=480, color=(120, 160, 90), fmt='JPEG', **kw):
    buf = io.BytesIO()
    Image.new('RGB', (w, h), color).save(buf, fmt, **kw)
    return buf.getvalue()


def bad_image(code, data):
    with pytest.raises(ValidationError) as e:
        load_image(data)
    assert e.value.code == code


def test_decision_mapping():
    cases = {(0, 0): 'accept', (0, 1): 'reject_contaminated', (1, 0): 'reject_wrong_type', (2, 1): 'retake_photo'}
    for (m, c), want in cases.items():
        pm = np.full((1, 3), 0.05)
        pm[0, m] = 0.9
        pc = np.full((1, 2), 0.05)
        pc[0, c] = 0.95
        d, _ = decide(pm, pc)
        assert DECISIONS[d[0]] == want
    pm = np.array([[0.9, 0.05, 0.05]])
    pc = np.array([[0.8, 0.2]])
    assert abs(decide(pm, pc)[1][0] - 0.72) < 1e-9      # plastic: material conf x cleanliness conf
    assert final_decision(pm, pc, 0.9)[0] == UNCERTAIN
    assert final_decision(pm, pc, 0.7)[0] == 'accept'


def test_temperature():
    p = np.array([[0.6, 0.3, 0.1]])
    assert np.allclose(apply_temperature(p, 1.0), p)
    assert apply_temperature(p, 0.5)[0, 0] > 0.6 > apply_temperature(p, 2.0)[0, 0]


def test_image_validation():
    img = load_image(image_bytes())
    assert img.mode == 'RGB' and max(img.size) == 640
    assert max(load_image(image_bytes(3000, 2000)).size) == 768       # downscaled like training
    assert load_image(image_bytes(fmt='PNG')).mode == 'RGB'
    bad_image('empty_file', b'')
    bad_image('unsupported_type', b'not an image at all' * 10)
    bad_image('unsupported_type', image_bytes(fmt='GIF'))
    bad_image('image_too_small', image_bytes(100, 80))
    bad_image('file_too_large', b'\xff' * (8 * 1024 * 1024 + 1))
    bad_image('corrupt_image', image_bytes(800, 600, quality=95)[:3000])
    big = io.BytesIO()
    Image.new('L', (8000, 6000)).save(big, 'PNG')                       # small file, 48M pixels
    bad_image('too_many_pixels', big.getvalue())
    bomb = io.BytesIO()
    Image.new('1', (20000, 20000)).save(bomb, 'PNG')                     # ~50 KB file, 400M pixels
    bad_image('too_many_pixels', bomb.getvalue())


def test_exif_rotation_applied_and_metadata_stripped():
    im = Image.new('RGB', (400, 200), (10, 200, 10))
    exif = im.getexif()
    exif[0x0112] = 6                                                     # "rotate 90" orientation tag
    buf = io.BytesIO()
    im.save(buf, 'JPEG', exif=exif)
    out = load_image(buf.getvalue())
    assert out.size == (200, 400)
    assert not out.getexif()


def test_photo_url_allow_list():
    hosts = {'abcd.supabase.co'}
    ok = 'https://abcd.supabase.co/storage/v1/object/sign/pickup-photos/p.jpg?token=x'
    assert photo_url_allowed(ok, hosts)
    assert photo_url_allowed(ok.replace('abcd', 'ABCD'), hosts)
    for bad in [
        'http://abcd.supabase.co/storage/v1/object/sign/p.jpg',            # not https
        'https://evil.example/storage/v1/object/sign/p.jpg',               # other host
        'https://abcd.supabase.co.evil.example/storage/v1/object/sign/p',  # look-alike host
        'https://user:pw@abcd.supabase.co/storage/v1/object/sign/p.jpg',  # credentials
        'https://abcd.supabase.co:8443/storage/v1/object/sign/p.jpg',      # odd port
        'https://abcd.supabase.co/rest/v1/riders',                         # other path
        'https://169.254.169.254/latest/meta-data/',                       # cloud metadata (SSRF)
        'file:///etc/passwd',
        'https://abcd.supabase.co:99999/storage/v1/object/sign/p.jpg',     # invalid port
    ]:
        assert not photo_url_allowed(bad, hosts), bad


def test_dhash_near_duplicates():
    rng = np.random.default_rng(0)
    base = Image.fromarray(rng.integers(0, 255, (300, 400, 3), dtype=np.uint8))
    other = Image.fromarray(rng.integers(0, 255, (300, 400, 3), dtype=np.uint8))
    assert (dhash(base) ^ dhash(base.resize((200, 150)))).bit_count() <= 5
    assert (dhash(base) ^ dhash(other)).bit_count() > 15


def test_auth_and_rate_limit():
    toks = parse_tokens('be:backend:' + 'b' * 32 + '; rv:reviewer:' + 'r' * 32)
    assert authenticate(toks, 'Bearer ' + 'b' * 32, 'verify') == 'be'
    assert authenticate(toks, 'Bearer ' + 'r' * 32, 'upload') == 'rv'
    for header, perm, status in [('Bearer ' + 'b' * 32, 'upload', 403), ('Bearer ' + 'b' * 32, 'metrics', 403),
                                 ('Bearer wrong', 'verify', 401), (None, 'verify', 401)]:
        with pytest.raises(AuthError) as e:
            authenticate(toks, header, perm)
        assert e.value.status == status
    for bad in ['x:backend:short', 'x:superuser:' + 'z' * 32]:
        with pytest.raises(ValueError):
            parse_tokens(bad)
    rl = RateLimiter(3)
    assert [rl.check('a', 1000 + i) for i in range(4)] == [0, 0, 0, 58]
    assert rl.check('a', 1061) == 0 and rl.check('b', 1000) == 0


def test_monitor():
    m = Monitor(baseline_hist=[0.1] * 10)
    for i in range(40):
        m.record({'outcome': 'predicted', 'decision': 'accept', 'reason': 'ok', 'confidence': 0.95 if i % 2 else 0.85,
                  'declared_type': 'pet' if i < 30 else 'hdpe', 'duplicate_of': 'x' if i < 4 else None,
                  'latency_ms': 100 + i, 'model_ms': 50})
    for _ in range(10):
        m.record({'outcome': 'unusable_photo', 'error': 'image_too_small'})
    m.record({'outcome': 'url_blocked'})
    r = m.metrics()
    assert r['predictions'] == 40 and r['unusable_share'] == 0.2 and r['duplicate_rate'] == 0.1
    assert r['deferred_share_under_engine_policy'] == 0.5                 # 0.85 < engine accept threshold 0.90
    assert r['confidence_by_declared_type']['pet']['n'] == 30
    assert r['confidence_psi_vs_offline'] > 0.2
    assert any('drift ALERT' in a for a in r['alerts']) and any('security' in a for a in r['alerts'])
    assert psi([10] * 10, [0.1] * 10) < 1e-9
    assert m.check_duplicate('s1', 0b1011, 'A') is None
    assert m.check_duplicate('s1', 0b1011, 'A') is None                 # a retry of the same verification
    assert m.check_duplicate('s1', 0b1011, 'B') == 'A'                  # same photo, another verification
    assert m.check_duplicate('s2', 0b1010, 'C') == 'A'                  # near-identical photo


def test_preprocess_matches_torchvision_rules():
    assert resize_shorter_side(Image.new('RGB', (768, 431)), 256).size == (456, 256)
    assert resize_shorter_side(Image.new('RGB', (431, 768)), 256).size == (256, 456)
    assert resize_shorter_side(Image.new('RGB', (500, 500)), 256).size == (256, 256)
    assert center_crop(Image.new('RGB', (456, 256)), 224).size == (224, 224)
    x = to_input(Image.new('RGB', (640, 480), (255, 0, 128)), 224, [0.5] * 3, [0.5] * 3)
    assert x.shape == (1, 3, 224, 224) and x.dtype == np.float32 and x.flags['C_CONTIGUOUS']
    assert np.allclose(x[0, :, 0, 0], [1.0, -1.0, (128 / 255 - 0.5) / 0.5], atol=1e-6)

In [ ]:
%%writefile tests/test_service.py
"""Contract tests for service.py against spec/classifier-contract.md, with a fake model
and a fake photo store. No web server, no model file, no network."""
import hashlib
import io
import json
import uuid
from uuid import UUID

import pytest
from PIL import Image
from pydantic import BaseModel, ConfigDict, Field

from monitoring import Monitor
from security import RateLimiter, parse_tokens
from service import ClassifierService, PhotoFetchError, PhotoTooLarge

BACKEND, REVIEWER = 'b' * 32, 'r' * 32
HOST = 'abcd.supabase.co'
URL = f'https://{HOST}/storage/v1/object/sign/pickup-photos/p1.jpg?token=SECRET-SIGNING-TOKEN'


class EngineReply(BaseModel):
    """Copy of the Engine's strict parser (backend/app/adapters/classifier.py, _Reply)."""
    model_config = ConfigDict(extra='ignore', strict=True)
    verification_id: UUID
    decision: str = Field(pattern='^(accept|reject|uncertain)$')
    confidence: float = Field(ge=0, le=1, allow_inf_nan=False)
    reason: str = Field(pattern='^(ok|contaminated|wrong_type|duplicate)$')
    model_version: str = Field(min_length=1, max_length=100)


class FakeVerifier:
    version = 'verifier-v1+abc123def456'
    weights_sha256 = 'f' * 64

    def __init__(self, suggested='accept', confidence=0.93, fail=False):
        self.suggested, self.confidence, self.fail, self.calls = suggested, confidence, fail, 0

    def predict(self, img):
        self.calls += 1
        if self.fail:
            raise RuntimeError('model crashed')
        return {'decision': self.suggested, 'suggested_decision': self.suggested, 'confidence': self.confidence,
                'status': 'auto', 'material': {'plastic': 0.9, 'non_plastic': 0.05, 'discard': 0.05},
                'cleanliness': {'clean': 0.9, 'contaminated': 0.1}}


def photo(color=(120, 160, 90), w=640, h=480):
    buf = io.BytesIO()
    Image.new('RGB', (w, h), color).save(buf, 'JPEG')
    return buf.getvalue()


def make(verifier=None, store=None, fetch=None, per_minute=1000):
    store = {URL: photo()} if store is None else store

    def default_fetch(url):
        if url not in store:
            raise PhotoFetchError('HTTP 404')
        return store[url]

    tokens = parse_tokens(f'engine:backend:{BACKEND};rv1:reviewer:{REVIEWER}')
    svc = ClassifierService(verifier or FakeVerifier(), Monitor(), tokens, RateLimiter(per_minute),
                            fetch or default_fetch, [HOST])
    return svc, store


def body(data=None, **overrides):
    data = photo() if data is None else data
    b = {'verification_id': str(uuid.uuid4()), 'request_id': str(uuid.uuid4()), 'declared_type': 'pet',
         'photo': {'url': URL, 'sha256': hashlib.sha256(data).hexdigest()},
         'captured_at': '2026-10-11T09:30:00+00:00', 'capture_source': 'rider'}
    b.update(overrides)
    return b


def call(svc, b, token=BACKEND, key='same'):
    if key == 'same':
        key = b.get('verification_id') if isinstance(b, dict) else None
    raw = json.dumps(b).encode() if isinstance(b, dict) else b
    return svc.verify_json(f'Bearer {token}' if token else None, key, raw)


@pytest.mark.parametrize('suggested,decision,reason', [
    ('accept', 'accept', 'ok'),
    ('reject_contaminated', 'reject', 'contaminated'),
    ('reject_wrong_type', 'reject', 'wrong_type'),
])
def test_decisions_follow_the_contract_and_pass_the_engines_parser(suggested, decision, reason):
    svc, _ = make(FakeVerifier(suggested, 0.87))
    b = body()
    status, resp = call(svc, b)
    assert status == 200
    reply = EngineReply.model_validate_json(json.dumps(resp))
    assert str(reply.verification_id) == b['verification_id']
    assert (reply.decision, reply.reason, reply.confidence) == (decision, reason, 0.87)


def test_nothing_recognisable_is_a_422_so_the_engine_asks_for_a_retake():
    svc, _ = make(FakeVerifier('retake_photo', 0.7))
    status, resp = call(svc, body())
    assert status == 422 and resp['reason'] == 'nothing_recognisable'


@pytest.mark.parametrize('data,reason', [(b'not an image' * 50, 'unsupported_type'), (None, 'image_too_small')])
def test_unusable_photos_are_422(data, reason):
    data = photo(w=100, h=80) if data is None else data
    svc, store = make()
    store[URL] = data
    status, resp = call(svc, body(data))
    assert status == 422 and resp['reason'] == reason


def test_photo_hash_mismatch_is_422():
    svc, _ = make()
    b = body()
    b['photo']['sha256'] = '0' * 64
    assert call(svc, b)[0] == 422


def test_photo_that_cannot_be_fetched_is_503():
    svc, _ = make(store={})
    assert call(svc, body()) == (503, {'error': 'photo_unavailable'})


def test_oversized_download_is_422():
    def fetch(url):
        raise PhotoTooLarge()
    svc, _ = make(fetch=fetch)
    assert call(svc, body())[0] == 422


@pytest.mark.parametrize('url', [
    'https://evil.example/storage/v1/object/sign/x.jpg',
    'https://169.254.169.254/latest/meta-data/',
    f'http://{HOST}/storage/v1/object/sign/x.jpg',
    f'https://{HOST}/rest/v1/households',
])
def test_links_outside_our_storage_are_refused_without_fetching(url):
    fetched = []

    def fetch(u):
        fetched.append(u)
        return photo()
    svc, _ = make(fetch=fetch)
    b = body()
    b['photo']['url'] = url
    assert call(svc, b) == (400, {'error': 'photo_url_not_allowed'})
    assert fetched == []
    assert svc.monitor.metrics()['outcomes']['url_blocked'] == 1


@pytest.mark.parametrize('mutate', [
    lambda b: b.pop('verification_id'),
    lambda b: b.update(verification_id='not-a-uuid'),
    lambda b: b.update(declared_type='glass'),
    lambda b: b.update(capture_source='stranger'),
    lambda b: b['photo'].update(sha256='XYZ'),
    lambda b: b.update(household_phone='+233200000000'),       # personal data must never cross the boundary
    lambda b: b.update(captured_at='yesterday'),
], ids=['missing-id', 'bad-uuid', 'bad-type', 'bad-source', 'bad-sha', 'extra-personal-field', 'bad-date'])
def test_malformed_requests_are_400_never_422(mutate):
    svc, _ = make()
    b = body()
    mutate(b)
    status, resp = call(svc, b, key=None)
    assert status == 400 and resp['error'] == 'invalid_request'


def test_invalid_json_is_400():
    svc, _ = make()
    assert call(svc, b'{not json', key=None) == (400, {'error': 'invalid_json'})


def test_idempotency_key_must_match_the_verification_id():
    svc, _ = make()
    assert call(svc, body(), key=str(uuid.uuid4())) == (400, {'error': 'idempotency_key_mismatch'})


def test_a_retry_gets_the_same_answer_without_running_the_model_again():
    v = FakeVerifier()
    svc, _ = make(v)
    b = body()
    first = call(svc, b)
    second = call(svc, b)
    assert first == second and v.calls == 1
    assert svc.monitor.metrics()['outcomes']['idempotent_replay'] == 1


def test_model_crash_is_500_and_not_cached():
    v = FakeVerifier(fail=True)
    svc, _ = make(v)
    b = body()
    assert call(svc, b)[0] == 500
    assert call(svc, b)[0] == 500 and v.calls == 2


def test_authentication_and_rate_limit():
    svc, _ = make(per_minute=2)
    assert call(svc, body(), token=None)[0] == 401
    assert call(svc, body(), token='x' * 32)[0] == 401
    assert call(svc, body())[0] == 200
    assert call(svc, body())[0] == 200
    status, resp = call(svc, body())
    assert status == 429 and resp['retry_after_s'] > 0


def test_upload_path_is_for_reviewers_only_and_returns_details():
    svc, _ = make()
    assert svc.verify_upload(f'Bearer {BACKEND}', photo())[0] == 403
    status, resp = svc.verify_upload(f'Bearer {REVIEWER}', photo(), 'pet')
    assert status == 200 and resp['decision'] == 'accept' and 'material' in resp


def test_reused_photo_shows_in_monitoring_but_is_not_rejected_here():
    svc, _ = make()
    assert call(svc, body())[1]['reason'] == 'ok'
    status, resp = call(svc, body())                 # same photo, new verification
    assert status == 200 and resp['reason'] == 'ok'  # the Engine already rejects reused photos
    assert svc.monitor.metrics()['duplicate_rate'] == 0.5


def test_metrics_and_logs_need_a_reviewer_and_never_contain_links_or_images():
    svc, _ = make()
    call(svc, body())
    assert svc.metrics(f'Bearer {BACKEND}')[0] == 403
    assert svc.metrics(f'Bearer {REVIEWER}')[0] == 200
    status, logs = svc.logs(f'Bearer {REVIEWER}', 10)
    dumped = json.dumps(logs)
    assert status == 200 and 'SECRET-SIGNING-TOKEN' not in dumped and URL not in dumped


def test_health():
    svc, _ = make()
    status, resp = svc.health()
    assert status == 200 and resp['model_version'].startswith('verifier-v1')

In [ ]:
%%writefile tests/test_http.py
"""HTTP-layer tests (main.py) with the model and photo fetcher replaced by fakes.
Skipped where FastAPI isn't installed; CI installs it from requirements.txt."""
import hashlib
import io
import json
import uuid

import pytest
from PIL import Image

pytest.importorskip('fastapi')
from fastapi.testclient import TestClient

import main
import photo_fetch
import verifier
from tests.test_service import FakeVerifier

BACKEND, REVIEWER = 'b' * 32, 'r' * 32
URL = 'https://abcd.supabase.co/storage/v1/object/sign/pickup-photos/p.jpg?token=x'


def jpeg():
    buf = io.BytesIO()
    Image.new('RGB', (640, 480), (120, 160, 90)).save(buf, 'JPEG')
    return buf.getvalue()


@pytest.fixture
def client(monkeypatch, tmp_path):
    data = jpeg()
    monkeypatch.setenv('SERVICE_TOKENS', f'engine:backend:{BACKEND};rv1:reviewer:{REVIEWER}')
    monkeypatch.setenv('PHOTO_URL_ALLOWED_HOSTS', 'abcd.supabase.co')
    monkeypatch.setenv('LOG_DIR', str(tmp_path))
    monkeypatch.setattr(verifier, 'fetch_model_files', lambda: ('w', 'c'))
    monkeypatch.setattr(verifier, 'Verifier', lambda w, c, sha=None: FakeVerifier())
    monkeypatch.setattr(photo_fetch, 'make_fetcher', lambda: (lambda url: data))
    with TestClient(main.app) as c:
        c.photo = data
        yield c


def request_body(data):
    return {'verification_id': str(uuid.uuid4()), 'request_id': str(uuid.uuid4()), 'declared_type': 'pet',
            'photo': {'url': URL, 'sha256': hashlib.sha256(data).hexdigest()},
            'captured_at': '2026-10-11T09:30:00+00:00', 'capture_source': 'rider'}


def test_health(client):
    assert client.get('/v1/health').status_code == 200
    assert client.get('/health').status_code == 200


def test_engine_call(client):
    b = request_body(client.photo)
    r = client.post('/v1/verify', json=b, headers={'Authorization': f'Bearer {BACKEND}',
                                                    'Idempotency-Key': b['verification_id']})
    assert r.status_code == 200 and r.json()['verification_id'] == b['verification_id']
    assert r.headers['cache-control'] == 'no-store'


def test_wrong_content_is_400_not_422(client):
    r = client.post('/v1/verify', content=b'garbage', headers={'Authorization': f'Bearer {BACKEND}'})
    assert r.status_code == 400
    r = client.post('/v1/verify-upload', headers={'Authorization': f'Bearer {REVIEWER}'})   # no file at all
    assert r.status_code == 400


def test_upload_and_monitoring(client):
    h = {'Authorization': f'Bearer {REVIEWER}'}
    r = client.post('/v1/verify-upload', headers=h, files={'image': ('p.jpg', client.photo, 'image/jpeg')},
                    data={'declared_type': 'pet'})
    assert r.status_code == 200 and 'material' in r.json()
    m = client.get('/metrics', headers=h)
    assert m.status_code == 200 and m.json()['predictions'] == 1
    assert client.get('/v1/logs?limit=5', headers=h).status_code == 200
    assert client.get('/metrics', headers={'Authorization': f'Bearer {BACKEND}'}).status_code == 403
    assert json.loads(client.get('/v1/logs', headers=h).text)['events']

In [ ]:
# 3. Tests and lint, exactly as CI runs them
!cd /content/intelligence && python -m pytest -q && ruff check . && echo "TESTS AND LINT PASSED"

In [ ]:
# 4. Export the PyTorch model to ONNX, then check that ONNX gives the same outputs
import torch, timm, torch.nn as nn, onnx, onnxruntime as ort
from torchvision import transforms as T
cfg = json.loads((FT / 'verifier_v1_config.json').read_text())
timm.layers.set_fused_attn(False)          # plain attention ops export cleanly to ONNX (same maths)

class VerifierV1(nn.Module):               # identical to the Stage 2c training notebook
    def __init__(self, backbone):
        super().__init__(); self.backbone = backbone; d = backbone.num_features
        self.material = nn.Linear(d, len(cfg['material_classes'])); self.clean = nn.Linear(d, len(cfg['clean_classes']))
    def forward(self, x):
        f = self.backbone(x); return self.material(f), self.clean(f)

tm = VerifierV1(timm.create_model(cfg['timm_id'], pretrained=False, num_classes=0))
tm.load_state_dict(torch.load(FT / 'verifier_v1.pt', map_location='cpu'), strict=True); tm.eval()

ONNX_PATH = DEPLOY / 'verifier_v1.onnx'
kw = dict(input_names=['image'], output_names=['material_logits', 'clean_logits'], opset_version=17,
          dynamic_axes={'image': {0: 'batch'}, 'material_logits': {0: 'batch'}, 'clean_logits': {0: 'batch'}})
dummy = torch.randn(1, 3, cfg['image_size'], cfg['image_size'])
for extra in ({'dynamo': False}, {}, {'dynamo': True}):
    try:
        torch.onnx.export(tm, dummy, str(ONNX_PATH), **kw, **extra); print('exported with', extra or 'defaults'); break
    except Exception as e:
        print('export attempt', extra, 'failed:', type(e).__name__, str(e)[:150])
onnx.checker.check_model(str(ONNX_PATH))
print(f'ONNX size: {ONNX_PATH.stat().st_size / 1e6:.1f} MB')

# Same input to both models -> outputs must match
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(cfg['image_size']), T.ToTensor(), T.Normalize(cfg['normalize_mean'], cfg['normalize_std'])])
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
from validation import load_image
from preprocess import to_input
TEST_FILES = sorted(f for d in sorted(TEST_DIR.iterdir()) for f in sorted(d.iterdir()))
max_out_diff, max_in_diff = 0.0, 0.0
with torch.no_grad():
    for f in TEST_FILES:
        img = load_image(f.read_bytes())
        xt = EVAL_TF(img).unsqueeze(0); xn = to_input(img, cfg['image_size'], cfg['normalize_mean'], cfg['normalize_std'])
        max_in_diff = max(max_in_diff, float(np.abs(xt.numpy() - xn).max()))
        tm_m, tm_c = tm(xt); on_m, on_c = sess.run(None, {'image': xt.numpy()})
        max_out_diff = max(max_out_diff, float(np.abs(tm_m.softmax(1).numpy() - torch.tensor(on_m).softmax(1).numpy()).max()),
                           float(np.abs(tm_c.softmax(1).numpy() - torch.tensor(on_c).softmax(1).numpy()).max()))
print(f'{len(TEST_FILES)} test images | preprocessing max difference {max_in_diff:.2e} | ONNX vs PyTorch max probability difference {max_out_diff:.2e}')
assert max_in_diff < 1e-4, 'numpy preprocessing differs from torchvision - training/serving skew'
assert max_out_diff < 1e-3, 'ONNX outputs differ from PyTorch'
print('ONNX and preprocessing checks PASSED')

In [ ]:
# 5. PARITY CHECK: the full service code (ONNX) must reproduce the Stage 2c offline evaluation
# Stage 2c on these 76 non-discard local test images: accept 14/14, contaminated 8/14, wrong type 18/48 = 40/76.
shutil.copy(FT / 'verifier_v1_config.json', DEPLOY / 'verifier_v1_config.json')
from verifier import Verifier, sha256_file
ver = Verifier(str(ONNX_PATH), str(DEPLOY / 'verifier_v1_config.json'))
TRUE = {'plastic_clean': 'accept', 'plastic_contaminated': 'reject_contaminated', 'non_plastic': 'reject_wrong_type'}
rows, lat = [], []
for f in TEST_FILES:
    img = load_image(f.read_bytes()); t0 = time.perf_counter(); out = ver.predict(img); lat.append((time.perf_counter() - t0) * 1000)
    rows.append({'true': TRUE[f.parent.name], 'suggested': out['suggested_decision'], 'conf': out['confidence'], 'status': out['status']})
correct = sum(r['true'] == r['suggested'] for r in rows)
print(f'Service (ONNX) on {len(rows)} local test images: {correct} correct ({correct / len(rows):.1%}); offline Stage 2c: 40 (52.6%)')
for t in TRUE.values():
    sub = [r for r in rows if r['true'] == t]; print(f'  {t:20s} {sum(r["suggested"] == t for r in sub)}/{len(sub)}')
PARITY_OK = abs(correct - 40) <= 3
print('PARITY', 'OK' if PARITY_OK else 'FAILED - do not deploy')
print(f'CPU latency (1 thread): median {np.median(lat):.0f} ms, p95 {np.percentile(lat, 95):.0f} ms')
counts = np.histogram([r['conf'] for r in rows], bins=10, range=(0, 1))[0]
BASELINE = (counts / counts.sum()).round(4).tolist()
print('Baseline confidence shares (for the live drift hook):', BASELINE)
assert PARITY_OK

In [ ]:
# 6. Final deployment files: ONNX + config with baseline + model card
cfg_d = dict(cfg, baseline_confidence_shares=BASELINE,
             baseline_source=f'ONNX service predictions on {len(rows)} non-discard local test images (parity check)',
             onnx_export={'opset': 17, 'max_prob_diff_vs_pytorch': max_out_diff, 'preprocessing_max_diff': max_in_diff})
(DEPLOY / 'verifier_v1_config.json').write_text(json.dumps(cfg_d, indent=2))
WEIGHTS_SHA = sha256_file(ONNX_PATH)
(DEPLOY / 'README.md').write_text(f'''# Waste verifier v1 (private)
ViT-S/16 backbone (Stage 1 selection, RealWaste) fine-tuned with a material head (plastic / non_plastic / discard)
and a cleanliness head (clean / contaminated) on RealWaste + human-labelled local Ghana images (Stage 2c).
Served as ONNX (`verifier_v1.onnx`, sha256 `{WEIGHTS_SHA}`); `verifier_v1.pt` is the PyTorch source.

- local test (113 images): decision accuracy 60.2%, pay/no-pay accuracy 91.2%, clean-plastic recall 100%
- auto-decision threshold {cfg["auto_decision_threshold"]:.3f}: model decides 45% alone at 78% accuracy; rest to human review
- intended use: assistant to a human verifier in the pilot; NOT for unsupervised payout decisions
- training data: research use only; images not redistributed
''')
print('ONNX sha256:', WEIGHTS_SHA)

In [ ]:
# 7. END-TO-END CONTRACT TEST with the real model (photo store replaced by a local dictionary)
import uuid, importlib
from fastapi.testclient import TestClient
import photo_fetch, main
HOST = 'test-project.supabase.co'
TOK = {'backend': secrets.token_urlsafe(32), 'reviewer': secrets.token_urlsafe(32)}
os.environ.update({'MODEL_DIR': str(DEPLOY), 'LOG_DIR': '/content/test_logs', 'EXPECTED_WEIGHTS_SHA256': WEIGHTS_SHA,
                   'PHOTO_URL_ALLOWED_HOSTS': HOST, 'RATE_LIMIT_PER_MIN': '1000',
                   'SERVICE_TOKENS': f"engine:backend:{TOK['backend']};rv1:reviewer:{TOK['reviewer']}"})
STORE = {}
photo_fetch.make_fetcher = lambda: (lambda url: STORE[url])
importlib.reload(main)
sample = TEST_FILES[0].read_bytes()
url = f'https://{HOST}/storage/v1/object/sign/pickup-photos/test.jpg?token=x'; STORE[url] = sample
def body(**kw):
    b = {'verification_id': str(uuid.uuid4()), 'request_id': str(uuid.uuid4()), 'declared_type': 'pet',
         'photo': {'url': url, 'sha256': hashlib.sha256(sample).hexdigest()},
         'captured_at': '2026-10-25T09:30:00+00:00', 'capture_source': 'rider'}
    b.update(kw); return b
E = lambda b: {'Authorization': f"Bearer {TOK['backend']}", 'Idempotency-Key': b['verification_id']}
R = {'Authorization': f"Bearer {TOK['reviewer']}"}
results = []
def check(name, resp, status):
    ok = resp.status_code == status; results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {resp.status_code} {resp.text[:120]}"); return resp
with TestClient(main.app) as c:
    check('health', c.get('/v1/health'), 200)
    b = body(); first = check('engine call, real model', c.post('/v1/verify', json=b, headers=E(b)), 200)
    reply = first.json()
    assert set(reply) == {'verification_id', 'decision', 'confidence', 'reason', 'model_version'}
    assert reply['verification_id'] == b['verification_id'] and reply['decision'] in ('accept', 'reject', 'uncertain')
    again = check('engine retry (same id) -> same answer', c.post('/v1/verify', json=b, headers=E(b)), 200)
    results.append(again.json() == reply)
    b2 = body(); b2['photo'] = dict(b2['photo'], url='https://169.254.169.254/latest/meta-data/')
    check('SSRF: link to cloud metadata', c.post('/v1/verify', json=b2, headers=E(b2)), 400)
    b3 = body(household_phone='+233200000000')
    check('personal data field in request', c.post('/v1/verify', json=b3, headers=E(b3)), 400)
    b4 = body(); b4['photo'] = dict(b4['photo'], sha256='0' * 64)
    check('photo hash mismatch', c.post('/v1/verify', json=b4, headers=E(b4)), 422)
    check('no token', c.post('/v1/verify', json=body()), 401)
    check('backend token on upload path', c.post('/v1/verify-upload', headers=E(body()), files={'image': ('a.jpg', sample, 'image/jpeg')}), 403)
    check('reviewer upload', c.post('/v1/verify-upload', headers=R, files={'image': ('a.jpg', sample, 'image/jpeg')}), 200)
    check('metrics (reviewer)', c.get('/metrics', headers=R), 200)
print(f'\n{sum(results)}/{len(results)} end-to-end checks passed')
assert all(results), 'fix failures before publishing'

In [ ]:
# 8. Upload the ONNX model to the PRIVATE Hugging Face model repo (the model registry)
from huggingface_hub import HfApi
api = HfApi(token=userdata.get('HF_WRITE').strip())
USER = api.whoami()['name']; MODEL_REPO = f'{USER}/waste-verifier-model'
api.create_repo(MODEL_REPO, repo_type='model', private=True, exist_ok=True)
commit = api.upload_folder(folder_path=str(DEPLOY), repo_id=MODEL_REPO, repo_type='model',
                           commit_message='verifier v1 ONNX export for Render deployment')
MODEL_REVISION = commit.oid
also_pt = api.upload_file(path_or_fileobj=str(FT / 'verifier_v1.pt'), path_in_repo='verifier_v1.pt', repo_id=MODEL_REPO,
                          commit_message='PyTorch source weights (provenance)') if not api.file_exists(MODEL_REPO, 'verifier_v1.pt') else None
print('Model repo (private):', MODEL_REPO, '| serving revision', MODEL_REVISION)
# copy the service folder to Drive so it is easy to upload to GitHub from your laptop
shutil.copytree(SVC, FT / 'intelligence', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', 'logs', '.pytest_cache', '.ruff_cache'))
print('Service code copied to Drive:', FT / 'model-service')

In [ ]:
# 9. Render settings + service tokens. Shown ONCE.
# 1) Copy the ENV block into Render (Environment -> 'Add from .env').
# 2) Save the per-caller tokens in a password manager.
# 3) Then CLEAR THIS CELL'S OUTPUT (click the output's ⋮ menu -> Clear output) so no secret stays in the notebook file.
CALLERS = [('backend', 'backend'), ('reviewer1', 'reviewer'), ('reviewer2', 'reviewer'), ('reviewer3', 'reviewer'), ('admin', 'admin')]
ISSUED = {n: secrets.token_urlsafe(32) for n, _ in CALLERS}
SUPABASE_HOST = input('Supabase project host (from SUPABASE_URL, e.g. abcd.supabase.co): ').strip().lower()
SUPABASE_HOST = SUPABASE_HOST.replace('https://', '').split('/')[0]
ENV = {'PHOTO_URL_ALLOWED_HOSTS': SUPABASE_HOST, 'SERVICE_TOKENS': ';'.join(f'{n}:{r}:{ISSUED[n]}' for n, r in CALLERS), 'MODEL_REPO': MODEL_REPO,
       'MODEL_REVISION': MODEL_REVISION, 'HF_TOKEN': userdata.get('HF_READ').strip(),
       'EXPECTED_WEIGHTS_SHA256': WEIGHTS_SHA, 'ORT_THREADS': '1', 'RATE_LIMIT_PER_MIN': '60'}
print('==== ENV block for Render (copy everything between the lines) ====')
for k, v in ENV.items(): print(f'{k}={v}')
print('==================================================================\n')
print('Per-caller tokens (backend -> backend developer privately; reviewer1-3 -> paired team; admin -> you):')
for n, _ in CALLERS: print(f'  {n:10s} {ISSUED[n]}')

## 10. In your browser: GitHub, then Render, then connect the backend

**A. GitHub (pull request).** Open the repo, go into the `intelligence` folder, then **Add file → Upload files**. Drag in **the contents** of the `intelligence` folder (from Drive `ics591_finetune/intelligence`, or the zip), including the `tests` and `notebooks` folders. Choose **Create a new branch … and start a pull request**. Wait for CI to pass (`ci-ok`), then merge.

**B. Render.** Choose **New → Web Service** and pick the repo, branch `main`. Then:
- **Name:** `classifier-staging`
- **Root Directory:** `intelligence`
- **Language:** Docker
- **Region:** Frankfurt (same as the Engine)
- **Instance type:** Free
- **Environment → Add from .env:** paste the cell 9 ENV block
- **Advanced → Health Check Path:** `/v1/health`
- **Auto-Deploy:** Off, because CI deploys tested commits

Click **Create**. When the log shows `Application startup complete`, copy the URL.

**C. CI deploys** (whoever has repo admin): Render → the service → Settings → **Deploy Hook**, copied into GitHub → Settings → Environments → **staging** → secret **`INTELLIGENCE_DEPLOY_HOOK`**.

**D. Connect the Engine** (backend developer). **Order matters:** the Engine refuses to start if `CLASSIFIER=http` has no URL or token. On `engine-staging`:
1. Set `CLASSIFIER_URL` to the URL from step B and `CLASSIFIER_TOKEN` to the **backend** token from cell 9.
2. **Then** change `CLASSIFIER` from `none` to `http`. In `render.yaml` this is hard-coded, so change it there too, in a pull request.

In [ ]:
# 11. Smoke-test the LIVE service
import requests
URL = input('Paste your Render service URL (https://...onrender.com): ').strip().rstrip('/')
for i in range(12):        # free services sleep when idle; the first request can take up to a minute
    try:
        h = requests.get(f'{URL}/v1/health', timeout=90); print('health', h.status_code, h.json()); break
    except Exception as e:
        print('waiting for the service to wake up...', type(e).__name__); time.sleep(15)
RV = {'Authorization': f"Bearer {ISSUED['reviewer1']}"}
t0 = time.time()
live = requests.post(f'{URL}/v1/verify-upload', headers=RV, files={'image': ('a.jpg', sample, 'image/jpeg')},
                     data={'declared_type': 'pet'}, timeout=120)
print(f'upload  {live.status_code} (round trip {time.time() - t0:.1f}s)\n', json.dumps(live.json(), indent=1))
import uuid
bad = {'verification_id': str(uuid.uuid4()), 'request_id': str(uuid.uuid4()), 'declared_type': 'pet',
       'photo': {'url': 'https://169.254.169.254/latest/meta-data/', 'sha256': '0' * 64},
       'captured_at': '2026-10-25T09:30:00+00:00', 'capture_source': 'rider'}
print('SSRF attempt ->', requests.post(f'{URL}/v1/verify', json=bad, headers={'Authorization': f"Bearer {ISSUED['backend']}"}, timeout=60).status_code, '(expect 400)')
print('no token     ->', requests.post(f'{URL}/v1/verify', json=bad, timeout=60).status_code, '(expect 401)')
print('metrics:', json.dumps(requests.get(f'{URL}/metrics', headers=RV, timeout=60).json(), indent=1)[:1800])

In [ ]:
# 12. Deployment record (NO secrets) -> Drive. Evidence for Sections B4 and C3.
record = {'deployed_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'platform': 'Render free web service (Docker)',
          'service_url': URL, 'contract': 'spec/classifier-contract.md', 'model_repo': MODEL_REPO, 'model_revision': MODEL_REVISION,
          'onnx_sha256': WEIGHTS_SHA, 'model_version': live.json().get('model_version'),
          'checks': {'onnx_vs_pytorch_max_prob_diff': max_out_diff, 'preprocessing_max_diff': max_in_diff,
                     'parity': f'{correct}/{len(rows)} correct vs 40/76 offline', 'end_to_end_checks': f'{sum(results)}/{len(results)}'},
          'colab_cpu_latency_ms': {'median': float(np.median(lat)), 'p95': float(np.percentile(lat, 95))},
          'callers': [n for n, _ in CALLERS]}
(FT / 'deployment_record.json').write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

In [ ]:
# 13. OPTIONAL: rotate all service tokens (if one leaks, or after the adversarial review window)
ROTATE = False
if ROTATE:
    ISSUED = {n: secrets.token_urlsafe(32) for n, _ in CALLERS}
    print('Replace SERVICE_TOKENS in Render (Environment tab) with:\n')
    print('SERVICE_TOKENS=' + ';'.join(f'{n}:{r}:{ISSUED[n]}' for n, r in CALLERS))
    print('\nRender restarts automatically; old tokens stop working. Clear this output afterwards.')